In [ ]:
class Value:
    def __init__(self, data, children=()):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(children)

    def __repr__(self) -> str:
        return f"Value({self.data})"
    
    def __add__(self, other):
        val = other if isinstance(other, Value) else Value(other)
        res = Value(self.data + val.data, children=(self, val))

        def _backward():
            self.grad += 1.0 * res.grad
            val.grad += 1.0 * res.grad
        res._backward = _backward
        return res
    
    def __radd__(self, other):
        return self + other
    
    def __mul__(self, other):
        val = other if isinstance(other, Value) else Value(other)
        res = Value(self.data * val.data, children=(self, val))

        def _backward():
            self.grad += val.data * res.grad
            val.grad += self.data * res.grad
        res._backward = _backward
        return res
    
    def __rmul__(self, other):
        return self * other

    def backward(self):
        self.grad = 1
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)
        topo.reverse()

        for node in topo:
            node._backward()



a = Value(2.0)
b = Value(3.0)

c = a + b
d = Value(10)

e = c * d

f = a * 5 + e

f.backward()

print('a grad', a.grad)